# Lecture 5 — Lab: build & train a classifier on a real dataset

In the sandbox you traced the **five steps of a training step** by hand on toy
numbers — forward, loss, `zero_grad`, `backward`, `step`. Now run that exact loop
*for real*, on a GPU, against **Fashion-MNIST**: 70,000 real 28x28 photos of
clothing (T-shirts, coats, sneakers, bags). It looks like MNIST but it is
genuinely harder — shirts, coats and pullovers blur into one another, so a model
that aces handwritten digits will sweat here.

This is the **spine** notebook for the labs that follow. The helpers we define
once below — `accuracy`, `evaluate`, `train`, `plot_curves` — are the same shape
the Lecture 6 CNN notebook reuses to *beat* the dense baseline you build here.

> **Runtime -> Change runtime type -> T4 GPU** before you start. The whole
> notebook runs in well under a minute on a T4.

## 1. Setup & GPU check

Imports, a reproducible seed, and the device. If this prints `cpu`, go set the
T4 runtime (above) and re-run — it still works on CPU, just slower.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets
from torchvision.transforms import ToTensor
import matplotlib.pyplot as plt

torch.manual_seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cpu":
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU, then re-run.")

### The reusable helpers (the spine)

Define these once and reuse them all course long. Read each one — it is the
PyTorch version of something you already built by hand.

- **`accuracy`** — exact-match rate: of the predictions, how many hit the right
  class? (`argmax` picks the predicted class, compare to the label, take the mean.)
- **`evaluate`** — the **validation pass** from the lecture: `model.eval()` +
  `torch.no_grad()`, a forward pass and a metric, **no `backward`, no `step`**.
  The two switches change *correctness*, not just speed.
- **`plot_curves`** — draw train vs val loss (and accuracy) per epoch. The
  *shape* of these curves is the single most useful diagnostic in deep learning;
  the gap between them is the story (under- vs over-fitting).

In [ ]:
def accuracy(logits, targets):
    """Exact-match rate: fraction of predictions whose argmax equals the label."""
    preds = logits.argmax(dim=1)
    return (preds == targets).float().mean().item()


@torch.no_grad()
def evaluate(model, loader, loss_fn, device):
    """One validation pass: forward + metrics, no backward, no optimizer step."""
    model.eval()                                  # eval mode: dropout off, BN uses running stats
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)         # data on the SAME device as the model
        logits = model(x)
        total_loss += loss_fn(logits, y).item() * len(y)
        correct += (logits.argmax(1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct / n            # (mean loss, accuracy)


def plot_curves(history):
    """Plot train/val loss and accuracy vs epoch from a train() history dict."""
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(11, 4))
    ax_loss.plot(epochs, history["train_loss"], "-o", label="train")
    ax_loss.plot(epochs, history["val_loss"], "-o", label="val")
    ax_loss.set(xlabel="epoch", ylabel="loss", title="Loss (the gap is the story)")
    ax_loss.legend()
    ax_acc.plot(epochs, history["train_acc"], "-o", label="train")
    ax_acc.plot(epochs, history["val_acc"], "-o", label="val")
    ax_acc.set(xlabel="epoch", ylabel="accuracy", title="Accuracy")
    ax_acc.legend()
    plt.tight_layout()
    plt.show()

## 2. Get the real data — Fashion-MNIST

`torchvision` downloads it for us (Colab has internet). It ships a **train** set
(60,000 images) and a **test** set (10,000). We carve a **validation** split off
the train set so we can tune against data the model never trains on, and keep the
test set sealed for the single number we report at the very end.

**What's real / messy about it:** these are real photographs of clothing, not
clean line drawings. Several classes genuinely overlap — a *shirt*, a *coat*, a
*pullover* and a *T-shirt* can look nearly identical at 28x28 grayscale — so
there is an irreducible amount of confusion no model can fully remove. That's the
real-data lesson this dataset teaches.

In [ ]:
# ToTensor() loads images as float tensors in [0, 1] with shape (1, 28, 28).
train_full = datasets.FashionMNIST(root="data", train=True, download=True, transform=ToTensor())
test_set = datasets.FashionMNIST(root="data", train=False, download=True, transform=ToTensor())

# Hold out 10,000 images from train as a validation split (never trained on).
val_size = 10_000
train_size = len(train_full) - val_size
train_set, val_set = random_split(
    train_full, [train_size, val_size],
    generator=torch.Generator().manual_seed(0),   # reproducible split
)

batch_size = 128
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)   # shuffle TRAIN every epoch
val_loader = DataLoader(val_set, batch_size=batch_size)                     # never shuffle what you measure on
test_loader = DataLoader(test_set, batch_size=batch_size)

CLASSES = train_full.classes   # ["T-shirt/top", "Trouser", "Pullover", ...]
print(f"train {len(train_set)} | val {len(val_set)} | test {len(test_set)}")
print("classes:", CLASSES)

## 3. Look at the data

Seeing the data *is* the real-dataset experience. We show a grid of sample images
with their labels, then a class histogram (Fashion-MNIST is balanced — 6,000
train images per class — so accuracy is a fair headline metric here).

In [ ]:
# A 4x6 grid of sample images with their class labels.
fig, axes = plt.subplots(4, 6, figsize=(10, 7))
for ax, idx in zip(axes.flat, torch.randperm(len(train_set))[:24]):
    img, label = train_set[idx]
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(CLASSES[label], fontsize=8)
    ax.axis("off")
plt.suptitle("Fashion-MNIST samples (real clothing photos)")
plt.tight_layout()
plt.show()

In [ ]:
# Class histogram over the training split -> roughly balanced.
labels = torch.tensor([train_full.targets[i] for i in train_set.indices])
counts = torch.bincount(labels, minlength=len(CLASSES))
plt.figure(figsize=(9, 3.5))
plt.bar(range(len(CLASSES)), counts.tolist())
plt.xticks(range(len(CLASSES)), CLASSES, rotation=45, ha="right")
plt.ylabel("# train images")
plt.title("Class distribution (balanced)")
plt.tight_layout()
plt.show()

## 4. Build the model — the dense baseline

This is the **fully-connected (dense) baseline**: flatten the 28x28 image into a
784-vector, one hidden layer of 256 ReLU units, then 10 output logits (one per
class). It mirrors exactly what you built from scratch in Lectures 2-4. No
softmax in the model — `CrossEntropyLoss` applies it internally, so the model
outputs **raw logits**.

In the Lecture 6 notebook a small CNN beats this baseline by *looking at local
pixel neighbourhoods* instead of flattening the spatial structure away. This is
the number to beat.

In [ ]:
# 🔧 Your turn — this mirrors what you built from scratch; read it (and try
# writing it yourself) before running.
class DenseClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),            # (B, 1, 28, 28) -> (B, 784)
            nn.Linear(784, 256),     # 784 inputs  -> 256 hidden units
            nn.ReLU(),               # nonlinearity
            nn.Linear(256, 10),      # 256 hidden  -> 10 class logits
        )

    def forward(self, x):
        return self.net(x)


model = DenseClassifier().to(device)   # move the model to the GPU once, up front
print(model)
n_params = sum(p.numel() for p in model.parameters())
print(f"{n_params:,} parameters")

## 5. Train — the five-step loop, for real

Here is the **spine** `train()` function. Inside it is the same five-step loop you
traced by hand, now with `.to(device)` on every batch and a validation pass each
epoch so we watch *generalization*, not just fitting:

```
pred = model(x)          # 1. forward
loss = loss_fn(pred, y)  # 2. loss
optimizer.zero_grad()    # 3. clear last batch's grads  <-- the step everyone forgets
loss.backward()          # 4. backward (autograd fills .grad)
optimizer.step()         # 5. update every weight
```

It also does **early stopping**: track the best validation loss, checkpoint the
model whenever it improves, and stop once `patience` epochs pass with no
improvement — then restore the best checkpoint (the model that *generalized* best,
not the last, over-trained one).

In [ ]:
def train(model, train_loader, val_loader, *, epochs=12, lr=0.1, weight_decay=0.0,
          patience=3, device=device, verbose=True):
    """Train with the five-step loop; track train/val loss+acc; early-stop on val loss."""
    model.to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val, best_state, bad = float("inf"), None, 0

    for epoch in range(1, epochs + 1):
        model.train()                                  # train mode: dropout/BN behave as training
        run_loss, run_correct, n = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)          # batch onto the SAME device as the model
            pred = model(x)                            # 1. forward
            loss = loss_fn(pred, y)                    # 2. loss
            optimizer.zero_grad()                      # 3. clear last batch's gradients
            loss.backward()                            # 4. backward: autograd fills .grad
            optimizer.step()                           # 5. nudge every weight downhill
            run_loss += loss.item() * len(y)
            run_correct += (pred.argmax(1) == y).sum().item()
            n += len(y)

        train_loss, train_acc = run_loss / n, run_correct / n
        val_loss, val_acc = evaluate(model, val_loader, loss_fn, device)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        if verbose:
            print(f"epoch {epoch:2d} | train_loss {train_loss:.3f} acc {train_acc:.3f}"
                  f" | val_loss {val_loss:.3f} acc {val_acc:.3f}")

        if val_loss < best_val:                        # improved -> checkpoint, reset patience
            best_val, bad = val_loss, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        elif (bad := bad + 1) >= patience:             # no improvement for `patience` epochs
            if verbose:
                print(f"early stopping at epoch {epoch} (best val_loss {best_val:.3f})")
            break

    if best_state is not None:
        model.load_state_dict(best_state)              # restore the best epoch, not the last
    return history

In [ ]:
history = train(model, train_loader, val_loader, epochs=12, lr=0.1)

### Quick sanity check (do this before trusting any run)

Before believing a training run, confirm the loop can **overfit a single batch**
down to near-zero loss. If it can't, the bug is in the *wiring* (a missing
`zero_grad`, a frozen parameter, a shape/label mismatch), not your data or model.
A correct loop drives this toward 0 because the model can simply memorize so few
points.

In [ ]:
sanity = DenseClassifier().to(device)
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.SGD(sanity.parameters(), lr=0.1)
xb, yb = next(iter(train_loader))                 # one tiny batch
xb, yb = xb.to(device), yb.to(device)
for step in range(200):                            # the same five steps, on one batch
    loss = loss_fn(sanity(xb), yb)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 50 == 0:
        print(f"step {step:3d}: loss={loss.item():.5f}")   # should march toward ~0

### 🔧 Diagnose & fix the broken run

The cell below is a **copy of the training loop with one line deleted**:
`optimizer.zero_grad()` is gone. Run it and watch what the loss does. PyTorch
*accumulates* `backward()` into `param.grad`, so without the reset the gradient
buffer grows batch after batch and `step()` lurches on a bloated, stale gradient.

**🔧 Your turn — find and fix it using the "won't learn" checklist:**

1. **Print the loss.** `print(loss.item())` each step — is it a sane number, or
   already huge / `NaN`? (A loss that climbs or refuses to fall is the symptom.)
2. **Check the gradients reach the weights, at a sane size.** Right after
   `loss.backward()`, look at a parameter's gradient — here it *grows* every batch
   instead of staying steady, the fingerprint of accumulation.
3. **Check shapes & dtypes.** `logits.shape == (batch, 10)` and
   `y.dtype == torch.long` — `CrossEntropyLoss` wants raw logits and integer labels.

Then add the missing line back in the right place (step 3 of the five:
`zero_grad -> backward -> step`) and confirm the loss falls like it should.

In [ ]:
# 🔧 BROKEN: optimizer.zero_grad() has been removed. Diagnose, then fix it.
broken = DenseClassifier().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(broken.parameters(), lr=0.1)

broken.train()
for step, (x, y) in enumerate(train_loader):
    if step >= 60:
        break
    x, y = x.to(device), y.to(device)
    pred = broken(x)                 # 1. forward
    loss = loss_fn(pred, y)          # 2. loss
    # optimizer.zero_grad()          # 3. clear gradients  <-- MISSING. why does this break training?
    loss.backward()                  # 4. backward
    optimizer.step()                 # 5. step
    if step % 10 == 0:
        # checklist: print the loss AND the size of an accumulating gradient
        g = broken.net[1].weight.grad.abs().mean().item()
        print(f"step {step:3d}: loss={loss.item():7.3f}  |grad|={g:.4f}  "
              f"logits={tuple(pred.shape)}  y.dtype={y.dtype}")

## 6. Evaluate

First the **loss curves** from the (good) run. Read the *gap* between train and
val: close and low together = a good fit; train falling while val turns back up =
overfitting (the dense net has plenty of capacity to start memorizing).

In [ ]:
plot_curves(history)

### Confusion matrix — *which* classes get confused

Accuracy is one number; a **confusion matrix** shows *where* the mistakes land.
`cm[true, pred]` counts how often a true class was predicted as each class — the
diagonal is correct, off-diagonal is confusion. Watch the upper-body garments:
**T-shirt/top, Pullover, Coat and Shirt** trade places constantly because they
look alike at 28x28. That overlap is real and irreducible — the dataset's lesson
that a single accuracy number hides *which* class hurts.

In [ ]:
@torch.no_grad()
def confusion_matrix(model, loader, n_classes, device):
    model.eval()
    cm = torch.zeros(n_classes, n_classes, dtype=torch.long)
    for x, y in loader:
        x = x.to(device)
        preds = model(x).argmax(1).cpu()
        for t, p in zip(y, preds):
            cm[t, p] += 1                       # cm[true, pred]
    return cm


cm = confusion_matrix(model, test_loader, len(CLASSES), device)

fig, ax = plt.subplots(figsize=(7.5, 6.5))
im = ax.imshow(cm, cmap="Blues")
ax.set(xticks=range(len(CLASSES)), yticks=range(len(CLASSES)),
       xlabel="predicted", ylabel="true", title="Confusion matrix (test set)")
ax.set_xticklabels(CLASSES, rotation=45, ha="right")
ax.set_yticklabels(CLASSES)
for i in range(len(CLASSES)):                   # annotate counts
    for j in range(len(CLASSES)):
        ax.text(j, i, int(cm[i, j]), ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=7)
fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

test_loss, test_acc = evaluate(model, test_loader, nn.CrossEntropyLoss(), device)
print(f"TEST accuracy: {test_acc:.4f}")

## 7. Experiment — modify & observe, then beat the baseline

Change **one thing at a time** and re-read the curves. Three to try:

1. **Normalize the inputs, or not.** `ToTensor()` already scales pixels to
   `[0, 1]`; standardizing to mean 0 / std 1 (Fashion-MNIST's train mean ~ 0.286,
   std ~ 0.353) often lets you train a touch faster / higher. Rebuild the loaders
   with the `Normalize` transform below and retrain — does val accuracy move?
2. **Learning rate too high.** Re-run `train(..., lr=2.0)`. Watch the loss
   **diverge** (climb, maybe `NaN`): the step overshoots the minimum every time.
   The fix, as always, is to drop the LR ~10x.
3. **Beat ~88% val accuracy.** The plain baseline lands around there. Try a bigger
   hidden layer, more epochs (early stopping protects you), a touch of
   `weight_decay=1e-4`, or the normalized inputs from (1). Fill the table below.

> Reminder: tune against **validation**; touch **test** only once, at the end.

In [ ]:
# Experiment 1 — standardized inputs. Rebuild loaders with a Normalize transform.
from torchvision.transforms import Compose, Normalize

norm_tf = Compose([ToTensor(), Normalize((0.286,), (0.353,))])   # Fashion-MNIST train mean/std
train_n = datasets.FashionMNIST(root="data", train=True, download=True, transform=norm_tf)
test_n = datasets.FashionMNIST(root="data", train=False, download=True, transform=norm_tf)
tr_n, va_n = random_split(train_n, [train_size, val_size],
                          generator=torch.Generator().manual_seed(0))
train_loader_n = DataLoader(tr_n, batch_size=batch_size, shuffle=True)
val_loader_n = DataLoader(va_n, batch_size=batch_size)

torch.manual_seed(0)
model_norm = DenseClassifier().to(device)
hist_norm = train(model_norm, train_loader_n, val_loader_n, epochs=12, lr=0.1)
plot_curves(hist_norm)

In [ ]:
# Experiment 2 — LR too high: watch it diverge (loss climbs / NaN), then fix by dropping ~10x.
torch.manual_seed(0)
diverge = DenseClassifier().to(device)
_ = train(diverge, train_loader, val_loader, epochs=4, lr=2.0, patience=99)

### Ablation table — fill in YOUR numbers

| Setting | Best val accuracy | Notes |
|---|---|---|
| Baseline (`lr=0.1`, raw `[0,1]` inputs) | _____ | the number to beat |
| Standardized inputs (Experiment 1) | _____ | faster? higher? |
| `lr=2.0` (Experiment 2) | _____ | diverges? |
| Your best (bigger hidden / +epochs / weight_decay) | _____ | what helped most? |

## 8. Reflect — report YOUR results

Answer from your own run (these are exactly the questions the lecture's
self-check asks):

1. **What final validation accuracy did your best model reach?** Did you clear
   the ~88% target, and what single change helped most — and *why* (relate it to
   the loss curves)?
2. **Which classes get confused the most?** Point to the off-diagonal cells in
   your confusion matrix. Why are these the hard pairs for a dense net at 28x28?
3. **How big is your train-val gap?** If train accuracy sits well above val,
   you're starting to **overfit** — which regularizer from the lecture (weight
   decay, dropout, early stopping, more data/augmentation) would you reach for
   first, and why?
4. **What did removing `zero_grad` do** to the loss in the broken-run cell, and
   how does gradient accumulation explain it?

Next: the **Lecture 6 CNN notebook** reuses these same helpers (`train`,
`evaluate`, `plot_curves`, `accuracy`) to beat this dense baseline by exploiting
the spatial structure your `Flatten` threw away.